# Pipeline Preprocessing Data ASEAN
Notebook ini menjalankan pipeline lengkap: baca data mentah → preprocessing → simpan hasil.

In [13]:
import sys
import os

# =====================================================================
# SEL 1: Deteksi otomatis root folder proyek 'Kelompok1-ML'
# =====================================================================

current_path = os.getcwd()

while current_path:
    if os.path.exists(os.path.join(current_path, 'scripts')):
        break
    parent = os.path.dirname(current_path)
    if parent == current_path:
        current_path = None
        break
    current_path = parent

if current_path:
    if current_path not in sys.path:
        sys.path.append(current_path)
    print(f'✅ Berhasil menghubungkan root folder proyek di: {current_path}')
else:
    raise RuntimeError(
        '❌ Folder scripts tidak ditemukan.\n'
        'Pastikan notebook berada di dalam folder proyek Kelompok1-ML.'
    )

✅ Berhasil menghubungkan root folder proyek di: c:\Users\fsyah\Kelompok1-ML


In [14]:
# =====================================================================
# SEL 2: Import + Force Reload fungsi dari scripts/preprocessing.py
# =====================================================================

import importlib
import scripts.preprocessing as prep_module

importlib.reload(prep_module)

from scripts.preprocessing import handle_missing, add_lag_features, preprocess_data, FEATURES, TARGET

print('✅ Import & reload berhasil.')
print(f'   TARGET  : {TARGET}')
print(f'   FEATURES: {FEATURES}')

✅ Import & reload berhasil.
   TARGET  : GDP_Growth
   FEATURES: ['GDP_Per_Capita', 'Population_Growth', 'Exports_pct', 'Imports_pct', 'Life_Expectancy', 'GDP_Growth_lag1', 'GDP_Growth_lag2', 'GDP_lag1']


In [15]:
# =====================================================================
# SEL 3: Tentukan path input & output
# =====================================================================

input_file  = os.path.join(current_path, 'data', 'raw',       'dataset_asean.csv')
output_file = os.path.join(current_path, 'data', 'processed', 'dataset_asean.csv')

if not os.path.exists(input_file):
    raise FileNotFoundError(
        f'❌ File input tidak ditemukan:\n   {input_file}\n'
        'Pastikan file CSV sudah ada di folder data/raw/'
    )

print(f'📂 Input  : {input_file}')
print(f'📂 Output : {output_file}')

📂 Input  : c:\Users\fsyah\Kelompok1-ML\data\raw\dataset_asean.csv
📂 Output : c:\Users\fsyah\Kelompok1-ML\data\processed\dataset_asean.csv


In [16]:
# =====================================================================
# SEL 4: Jalankan pipeline preprocessing
# =====================================================================

print('\nMemulai proses pipeline data ASEAN...')
df_final = preprocess_data(input_file, output_file)

if df_final is None:
    raise ValueError(
        '❌ preprocess_data() mengembalikan None.\n'
        'Periksa scripts/preprocessing.py — pastikan ada return df di akhir fungsi.'
    )

print('\n✅ Pipeline selesai!')


Memulai proses pipeline data ASEAN...
Membaca data mentah dari: c:\Users\fsyah\Kelompok1-ML\data\raw\dataset_asean.csv
  Shape awal          : 650 baris x 8 kolom
  Menangani missing values (interpolasi per negara)...
  Kolom setelah handle_missing: ['Country', 'Year', 'GDP_Growth', 'GDP_Per_Capita', 'Population_Growth', 'Exports_pct', 'Imports_pct', 'Life_Expectancy']
  Membuat fitur lag (lag1, lag2)...
  Baris dihapus (NaN lag awal): 83
  Shape akhir         : 567 baris x 11 kolom
  Data berhasil disimpan ke: c:\Users\fsyah\Kelompok1-ML\data\processed\dataset_asean.csv

✅ Pipeline selesai!


In [17]:
# =====================================================================
# SEL 5: Pratinjau hasil data
# =====================================================================

print(f'Shape data akhir : {df_final.shape[0]} baris x {df_final.shape[1]} kolom')
print(f'Negara           : {sorted(df_final["Country"].unique().tolist())}')
print(f'Rentang tahun    : {df_final["Year"].min()} — {df_final["Year"].max()}')
print('\n5 baris pertama:')
df_final.head()

Shape data akhir : 567 baris x 11 kolom
Negara           : ['Brunei', 'Cambodia', 'Indonesia', 'Laos', 'Malaysia', 'Philippines', 'Singapore', 'Thailand', 'Vietnam']
Rentang tahun    : 1963 — 2025

5 baris pertama:


,Country,Year,GDP_Growth,GDP_Per_Capita,Population_Growth,Exports_pct,Imports_pct,Life_Expectancy,GDP_Growth_lag1,GDP_Growth_lag2,GDP_lag1
2,Brunei,1963,-0.645292,1028.884503,4.488367,41.680682,8.031846,60.625,-0.645292,-0.645292,1028.884503
3,Brunei,1964,-0.645292,1028.884503,4.600944,41.680682,8.031846,61.267,-0.645292,-0.645292,1028.884503
4,Brunei,1965,-0.645292,1028.884503,4.536670,41.680682,8.031846,61.721,-0.645292,-0.645292,1028.884503
5,Brunei,1966,-0.645292,1145.705922,4.443372,41.680682,8.031846,62.401,-0.645292,-0.645292,1028.884503
6,Brunei,1967,-0.645292,1148.843032,4.342764,41.680682,8.031846,62.909,-0.645292,-0.645292,1145.705922


In [18]:
# =====================================================================
# SEL 6: Cek missing values setelah preprocessing
# =====================================================================

missing = df_final[FEATURES + [TARGET]].isnull().sum()
print('Missing values per kolom (seharusnya semua 0):')
print(missing)

if missing.sum() == 0:
    print('\n✅ Tidak ada missing values — data siap untuk modeling!')
else:
    print('\n⚠️  Masih ada missing values. Periksa kembali fungsi handle_missing().')

Missing values per kolom (seharusnya semua 0):
GDP_Per_Capita       0
Population_Growth    0
Exports_pct          0
Imports_pct          0
Life_Expectancy      0
GDP_Growth_lag1      0
GDP_Growth_lag2      0
GDP_lag1             0
GDP_Growth           0
dtype: int64

✅ Tidak ada missing values — data siap untuk modeling!


In [19]:
# =====================================================================
# SEL 7: Statistik deskriptif fitur
# =====================================================================

print('Statistik deskriptif fitur + target:')
df_final[FEATURES + [TARGET]].describe().round(3)

Statistik deskriptif fitur + target:


,GDP_Per_Capita,Population_Growth,Exports_pct,Imports_pct,Life_Expectancy,GDP_Growth_lag1,GDP_Growth_lag2,GDP_lag1,GDP_Growth
count,567.000,567.000,567.000,567.000,567.000,567.000,567.000,567.000,567.000
mean,7098.367,1.974,52.926,51.419,65.726,4.784,4.778,6818.796,4.797
std,14273.612,1.299,48.740,43.891,10.286,4.648,4.659,13809.190,4.635
min,53.205,-8.404,2.764,5.325,11.295,-34.809,-34.809,53.205,-34.809
25%,393.665,1.369,20.026,21.415,60.898,3.440,3.429,365.956,3.463
50%,1226.437,1.994,38.155,38.175,68.061,5.092,5.101,1126.840,5.101
75%,4717.290,2.551,66.580,63.429,72.950,7.315,7.329,4298.154,7.278
max,90674.067,8.098,228.994,208.931,83.595,24.338,24.338,90674.067,24.338
